# Part 2: Sampling (Problems 010–016)

Once the transformer produces **logits** (one per vocabulary token), we need a strategy to pick the next token.

This notebook covers all the standard strategies:

- **Greedy**: always pick the highest-probability token
- **Temperature**: scale logits before softmax to sharpen or flatten the distribution
- **Top-K**: only sample from the K most likely tokens
- **Top-P (Nucleus)**: sample from the smallest set of tokens covering P% of the probability mass
- **Autoregressive generation**: loop to produce a full sequence

## Cell 1: Greedy vs temperature sampling side by side

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Make sure the project root is on sys.path so solutions/ is importable
project_root = Path('__file__').parent.parent if '__file__' in dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Also try the current directory's parent
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'solutions').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break


In [ ]:
import importlib
import torch
import torch.nn.functional as F

try:
    _m = importlib.import_module("solutions.010_greedy_sample")
    greedy_sample = _m.greedy_sample
except Exception:
    print("Solve problem 010 first:")
    print("  cp problems/010_greedy_sample.py solutions/010_greedy_sample.py")
    greedy_sample = None

try:
    _m = importlib.import_module("solutions.011_temperature_scaling")
    temperature_scaling = _m.temperature_scaling
except Exception:
    print("Solve problem 011 first:")
    print("  cp problems/011_temperature_scaling.py solutions/011_temperature_scaling.py")
    temperature_scaling = None

# Toy vocabulary of 10 tokens
vocab = ["the", "cat", "sat", "on", "mat", "dog", "fox", "ran", "home", "fast"]

torch.manual_seed(7)
# Logits: 'cat' (index 1) and 'dog' (index 5) are the top candidates
logits = torch.tensor([-1.0, 3.5, 0.2, -0.5, 0.1, 2.8, 0.5, -1.2, 0.3, -0.8])

probs = F.softmax(logits, dim=-1)
print("Token probabilities (before any sampling strategy):")
for i, (tok, p) in enumerate(zip(vocab, probs.tolist())):
    bar = "#" * int(p * 40)
    print(f"  [{i:2d}] '{tok:6s}'  {p:.3f}  {bar}")

print()

if greedy_sample is not None:
    greedy_idx = greedy_sample(logits)
    print(f"Greedy pick: [{greedy_idx}] '{vocab[greedy_idx]}' (always the argmax)")

if temperature_scaling is not None:
    print()
    for temp in [0.2, 0.8, 1.0, 2.0]:
        scaled_logits = temperature_scaling(logits.clone(), temp)
        scaled_probs = F.softmax(scaled_logits, dim=-1)
        top_idx = scaled_probs.argmax().item()
        entropy = -(scaled_probs * (scaled_probs + 1e-10).log()).sum().item()
        print(f"  T={temp:.1f}  →  top token: '{vocab[top_idx]}'  "
              f"(top prob: {scaled_probs[top_idx]:.3f}, entropy: {entropy:.3f})")

## Cell 2: Visualise how top-k filtering changes the probability distribution

In [ ]:
import importlib
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

try:
    _m = importlib.import_module("solutions.012_top_k_filter")
    top_k_filter = _m.top_k_filter
except Exception:
    print("Solve problem 012 first:")
    print("  cp problems/012_top_k_filter.py solutions/012_top_k_filter.py")
    top_k_filter = None

vocab = ["the", "cat", "sat", "on", "mat", "dog", "fox", "ran", "home", "fast"]
logits = torch.tensor([-1.0, 3.5, 0.2, -0.5, 0.1, 2.8, 0.5, -1.2, 0.3, -0.8])

x = np.arange(len(vocab))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

original_probs = F.softmax(logits, dim=-1).numpy()
axes[0].bar(x, original_probs, color="steelblue")
axes[0].set_title("Original distribution (no filter)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(vocab, rotation=45, ha="right")
axes[0].set_ylabel("Probability")

if top_k_filter is not None:
    for ax, k in zip(axes[1:], [3, 1]):
        filtered = top_k_filter(logits.clone(), k)
        # Replace -inf with very negative value for display
        display = filtered.clone()
        display[display == float('-inf')] = -1e9
        probs = F.softmax(display, dim=-1).numpy()
        colors = ["tomato" if p > 1e-6 else "lightgray" for p in probs]
        ax.bar(x, probs, color=colors)
        ax.set_title(f"After Top-K={k} filter")
        ax.set_xticks(x)
        ax.set_xticklabels(vocab, rotation=45, ha="right")
        ax.set_ylabel("Probability")
else:
    for ax, k in zip(axes[1:], [3, 1]):
        ax.set_title(f"Top-K={k} (solve problem 012)")
        ax.text(0.5, 0.5, "Not yet implemented",
                ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("/tmp/top_k_filter.png", dpi=100)
plt.show()
print("Top-K zeroes out all tokens outside the top-K, then renormalises.")

## Cell 3: Visualise how top-p nucleus filtering works

In [ ]:
import importlib
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

try:
    _m = importlib.import_module("solutions.013_top_p_nucleus_filter")
    top_p_nucleus_filter = _m.top_p_nucleus_filter
except Exception:
    print("Solve problem 013 first:")
    print("  cp problems/013_top_p_nucleus_filter.py solutions/013_top_p_nucleus_filter.py")
    top_p_nucleus_filter = None

vocab = ["the", "cat", "sat", "on", "mat", "dog", "fox", "ran", "home", "fast"]
logits = torch.tensor([-1.0, 3.5, 0.2, -0.5, 0.1, 2.8, 0.5, -1.2, 0.3, -0.8])
probs = F.softmax(logits, dim=-1)

# Sort by probability descending
sorted_probs, sorted_idx = probs.sort(descending=True)
sorted_vocab = [vocab[i] for i in sorted_idx.tolist()]
cumulative = sorted_probs.cumsum(dim=-1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: cumulative probability plot
x = np.arange(len(vocab))
axes[0].bar(x, sorted_probs.numpy(), color="steelblue", label="Individual")
ax2 = axes[0].twinx()
ax2.plot(x, cumulative, 'r-o', label="Cumulative")
ax2.axhline(y=0.9, color="orange", linestyle="--", label="p=0.90")
ax2.set_ylabel("Cumulative probability")
axes[0].set_xticks(x)
axes[0].set_xticklabels(sorted_vocab, rotation=45, ha="right")
axes[0].set_title("Sorted probabilities + cumulative sum")
axes[0].set_ylabel("Probability")
ax2.legend(loc="center right")

# Right: after top-p filter
if top_p_nucleus_filter is not None:
    for p_thresh in [0.9]:
        filtered = top_p_nucleus_filter(logits.clone(), p_thresh)
        display = filtered.clone()
        display[display == float('-inf')] = -1e9
        filtered_probs = F.softmax(display, dim=-1).numpy()
        colors = ["tomato" if filtered_probs[i] > 1e-6 else "lightgray"
                  for i in range(len(vocab))]
        axes[1].bar(x, probs.numpy(), color=colors)
        axes[1].set_title(f"After Top-P=0.90 nucleus filter\n(gray = excluded)")
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(vocab, rotation=45, ha="right")
        axes[1].set_ylabel("Probability")
else:
    axes[1].set_title("Top-P filter (solve problem 013)")
    axes[1].text(0.5, 0.5, "Not yet implemented",
                 ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig("/tmp/top_p_filter.png", dpi=100)
plt.show()
print("Nucleus sampling: keep the smallest set of tokens whose cumulative")
print("probability >= p, then renormalise. Adapts dynamically to the distribution.")

## Cell 4: Full generation loop — encode → generate 20 tokens → decode

In [ ]:
import importlib
import torch

try:
    _m = importlib.import_module("solutions.015_autoregressive_generate")
    autoregressive_generate = _m.autoregressive_generate
except Exception:
    print("Solve problem 015 first:")
    print("  cp problems/015_autoregressive_generate.py solutions/015_autoregressive_generate.py")
    autoregressive_generate = None

try:
    _m = importlib.import_module("solutions.001_build_token_vocab")
    build_token_vocab = _m.build_token_vocab
except Exception:
    build_token_vocab = None

try:
    _m = importlib.import_module("solutions.002_encode_and_decode")
    encode = _m.encode
    decode = _m.decode
except Exception:
    encode = decode = None

corpus = (
    "the quick brown fox jumps over the lazy dog "
    "the dog sat on the mat the cat ran to the fox "
    "the fox jumped over the gate the gate was tall "
    "a tall dog and a quick cat ran home the home was warm"
)

if all(fn is not None for fn in [build_token_vocab, encode, decode, autoregressive_generate]):
    token_to_id, id_to_token = build_token_vocab(corpus)
    vocab_size = len(token_to_id)

    prompt = "the quick brown"
    prompt_ids = encode(prompt, token_to_id)

    print(f"Prompt      : '{prompt}'")
    print(f"Prompt IDs  : {prompt_ids}")
    print(f"Vocab size  : {vocab_size}")
    print()

    # autoregressive_generate takes token_ids tensor and a model callable
    # We use a simple random model for demonstration
    def toy_model(ids):
        """Returns uniform random logits over the vocabulary."""
        return torch.randn(vocab_size)

    input_ids = torch.tensor(prompt_ids)
    generated_ids = autoregressive_generate(
        input_ids=input_ids,
        model=toy_model,
        max_new_tokens=20,
        temperature=0.9,
        top_k=5,
    )

    generated_tokens = decode(generated_ids.tolist(), id_to_token)
    print(f"Generated   : '{generated_tokens}'")
    print(f"Total tokens: {len(generated_ids)}"
          f" ({len(prompt_ids)} prompt + {len(generated_ids) - len(prompt_ids)} new)")
else:
    print("Complete problems 001, 002, and 015 to run the full generation loop.")